# 1 · What Are Code Clones?

*Part of the [ensemble-codesim](https://github.com/jorge-martinez-gil/ensemble-codesim) tutorial series — companion to Martinez-Gil (2025), "Advanced Detection of Source Code Clones via an Ensemble of Unsupervised Similarity Measures", SWQD 2025.*

**Code clones** are fragments of source code that are identical or similar to one another. Studies repeatedly find that **7–23%** of the code in a typical system is cloned, usually because developers copy-paste-and-adapt.

Why should you care?

- **Software maintenance** — a bug fixed in one copy must be fixed in every other copy; missed copies are a classic source of recurring defects.
- **Refactoring** — finding clones reveals where to introduce a shared function or class.
- **Plagiarism & academic integrity** — clone detection is the engine behind source-code plagiarism checkers (the `IR-Plag` / Karnalim dataset in this repo comes from exactly this setting).
- **Licensing & provenance** — detecting copied code helps track license obligations.
- **Code search & reuse** — "find me code similar to this" is clone detection in disguise.

### Learning objectives
By the end of this notebook you will be able to:
1. Define the four standard clone types and recognise each in real code.
2. Explain why **non-clones can look deceptively similar** (hard negatives).
3. Compute your first similarity score and see why **identifier normalisation** matters.

> Everything here runs with only **Python + NumPy** — no GPU, no model downloads.

In [ ]:
# --- setup: make the tutorial modules importable from anywhere in the repo ---
import os, sys
def _find_tutorials(start):
    d = os.path.abspath(start)
    for _ in range(8):
        if os.path.exists(os.path.join(d, "codesim_edu.py")):
            return d
        if os.path.exists(os.path.join(d, "tutorials", "codesim_edu.py")):
            return os.path.join(d, "tutorials")
        nd = os.path.dirname(d)
        if nd == d: break
        d = nd
    raise RuntimeError("Run this notebook from inside the ensemble-codesim repo.")
TUT  = _find_tutorials(os.getcwd())
REPO = os.path.dirname(TUT)
DATA = os.path.join(REPO, "outputs")
sys.path.insert(0, TUT); sys.path.insert(0, os.path.join(TUT, "examples"))
print("tutorials:", TUT)

In [ ]:
from clone_pairs import load_pairs

pairs = load_pairs()
print(f"Loaded {len(pairs)} labelled example pairs.\n")

def show_pair(p):
    bar = "=" * 70
    print(bar)
    ct = p["clone_type"]
    kind = f"Type-{ct}" if ct is not None else "NOT A CLONE (hard negative)"
    print(f"id={p['id']}   |   {kind}   |   label={p['label']}   |   lang={p['language']}")
    print(bar)
    print(">>> code A:\n" + p["code_a"])
    print(">>> code B:\n" + p["code_b"])
    print("WHY:", p["explanation"])
    print()

# index pairs by id for convenience
by_id = {p["id"]: p for p in pairs}
print("Available ids:")
for pid in by_id: print("  -", pid)

## The four clone types

The community taxonomy (Bellon et al. 2007; Roy & Cordy 2007) ranks clones by how much they differ:

| Type | Name | What changed |
|------|------|--------------|
| **1** | Exact | Only whitespace, layout, and comments |
| **2** | Renamed / parameterised | Type-1 **+** renamed identifiers, changed literals/types |
| **3** | Near-miss / gapped | Type-2 **+** added / removed / modified statements |
| **4** | Semantic | Same behaviour, **different** implementation (e.g. loop vs. recursion) |

Type-1 is easy for almost any tool; **Type-4 is the frontier** — it usually requires understanding *what the code does*, not just how it looks.

### Type-1 — exact clone

In [ ]:
show_pair(by_id["java_type1_sum"])

### Type-2 — renamed clone

Same structure, different names. A tool that compares raw text will see *less* overlap here than a human does.

In [ ]:
show_pair(by_id["java_type2_sum_renamed"])

### Type-3 — near-miss / gapped clone

A copy that was edited: statements inserted or removed. The "gap" is what makes detection hard.

In [ ]:
show_pair(by_id["java_type3_sum_gapped"])

### Type-4 — semantic clone

The two snippets compute the **same result** but share almost no surface structure. This is the hardest and most valuable case.

In [ ]:
show_pair(by_id["java_type4_factorial_recursive"])
show_pair(by_id["crosslang_type4_gcd"])  # bonus: the same algorithm in two languages

## Hard negatives: when *non*-clones look similar

Clone detection is not just about scoring clones highly — it is equally about **not** flagging code that merely *looks* alike. These "hard negatives" are where naive token-overlap measures produce false positives.

In [ ]:
show_pair(by_id["java_negative_sum_vs_max"])
show_pair(by_id["python_negative_boilerplate"])

## A first similarity score

Let us compute two simple measures for every pair:

- `jaccard_tokens` — overlap of the **raw** token sets.
- `normalized_jaccard` — the same, but **after replacing user identifiers with a placeholder** (so `addUp(values)` and `sum(a)` both become `ID(ID)`).

Watch what happens to the **Type-2** pairs.

In [ ]:
from codesim_edu import jaccard_tokens, normalized_jaccard

print(f"{'pair id':34s} {'type':5s} {'label':5s} {'jaccard':>8s} {'norm_jacc':>10s}")
print("-" * 66)
for p in pairs:
    ct = p["clone_type"]
    t = f"T{ct}" if ct is not None else "neg"
    j = jaccard_tokens(p["code_a"], p["code_b"])
    nj = normalized_jaccard(p["code_a"], p["code_b"])
    print(f"{p['id']:34s} {t:5s} {p['label']:<5d} {j:8.2f} {nj:10.2f}")

### What did we just see?

- For **Type-2** pairs, `jaccard_tokens` drops (different names => fewer shared tokens) while `normalized_jaccard` jumps back up to ~1.0. **Normalisation recovers renamed clones.**
- For the **hard negatives**, *both* measures can still be fairly high — overlap alone cannot tell `sum` from `max`. We will need more than one signal.
- For **Type-4**, even normalisation will not save a purely lexical measure (try editing the loop above to print Type-4 scores).

This tension — every measure has a blind spot — is exactly what motivates an **ensemble**.

### Exercise
Add your own pair to `tutorials/examples/clone_pairs.py` (say, a Type-3 clone in a language you like), re-run `load_pairs()`, and predict its two scores before printing them.

**Next:** [`02_similarity_measures.ipynb`](02_similarity_measures.ipynb) — the full family of measures and where each one fails.